# 2 Byte-Pair Encoding (BPE) Tokenizer

### Problem (unicode1): Understanding Unicode

In [1]:
# (a) '\x00'. There's not a single pre-defined symbol representing character chr(0), so it's represented by the hex representation of its code point
chr(0)

'\x00'

In [2]:
# (b): this is a string whose content is the printed representation
chr(0).__repr__()

"'\\x00'"

In [3]:
# (c) this is the string representation of the value
chr(0)

'\x00'

In [4]:
print(chr(0)) # nothing is printed because there's no symbol corresponding to \x00

 


In [5]:
"this is a test" + chr(0) + "string" # the value of the string, where \x00 is used

'this is a test\x00string'

In [6]:
print("this is a test" + chr(0) + "string") # again, since there's no symbol corresponding to \x00, it's not printed

this is a test string


### Problem (unicode 2): Unicode Encodings

In [7]:
# (a) UTF-8 use much fewer bytes to encode string than UTF-16 and UTF-32

test_string = "你好hello"
print(f"utf-8: {test_string.encode("utf-8")}")
print(f"utf-8 len: {len(test_string.encode("utf-8"))}")
print(f"utf-16 len: {len(test_string.encode("utf-16"))}")
print(f"utf-32 len: {len(test_string.encode("utf-32"))}")

utf-8: b'\xe4\xbd\xa0\xe5\xa5\xbdhello'
utf-8 len: 11
utf-16 len: 16
utf-32 len: 32


In [8]:
# (b) Some unicode characters are represented by multiple bytes. This function treats each byte as a single character, which doesn't handle multi-byte characters correctly.

def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

decode_utf8_bytes_to_str_wrong(b'\xe4\xbd\xa0\xe5\xa5\xbdhello')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe4 in position 0: unexpected end of data

In [10]:
# (c) According to utf-8 spec, byte \xe4 must be followed by a byte whos binary representation starts with prefix bits 10
b'\xe4\xe4'.decode("utf-8")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe4 in position 0: invalid continuation byte

### Problem (train_bpe_tinystories): BPE Training on TinyStories

#### Part (a) 
It took 46 seconds, with 42 seconds on pretokenization and 3 seconds on tokenization. Peak memory usage = 152M. 

The following tokens are the longest in the vocab, each with 15 bytes: "Ġresponsibility", "Ġaccomplishment", "Ġdisappointment". They seem to make sense because they correspond to " responsibility", " accomplishment" and " disappointment" in English.

#### Part (b)

Based on the scalene profiling below, the step of using regex to break documents into tokenizations took the most of time.

![Alt Text](tinystories_profile.png)


### Problem (train_bpe_expts_owt): BPE Training on OpenWebText

#### Part (a)

The longest tokens in the vocabulary are "ÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤ" and "----------------------------------------------------------------", each with 64 bytes.

They seem to make sense because the former corresponds to "ÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ", which seem to appear in the text as separators/fillers a lot, and the latter is a common separator.

#### Part (b)

Similarities: both are dominated by English words

Contrasts: owt vocab is more diverse, with more unicode phrases (still rare though) and lots of variants of separators

### Problem (tokenizer_experiments): Experiments with tokenizers

#### Part (a)

- TinyStories: 7552 bytes tokenized into 1815 tokens (4.16 bytes/token)
- OpenWebText: 31604 bytes tokenized into 6721 tokens (4.7 bytes/token)

#### Part (b)
- When using TinyStories tokenizer, the OpenWebText sample was tokenized into 9882 tokens, leading to a lower compression ratio of 3.2 bytes/token

#### Part (c)

It took 1.25 seconds to tokenize the OpenWebText sample of 31604 bytes. So, the throughput is 25283 bytes / second.

To tokenize Pile dataset, it'd take 825_000_000_000 / 25283 = 32630621 seconds = 9064 hours = 377 days. Since the tokenizer uses a cache, the actual time taken for processing a large corpus is probably shorter.

In [11]:
seconds = 825_000_000_000 / 25283
hours = seconds / 3600
days = hours / 24
print(seconds)
print(hours)
print(days)

32630621.366135348
9064.061490593152
377.66922877471467


#### Part (d)

The bigger vocab size between the two is 32000, so the token ids are all in the range of [0, 2^16) and hence can fit in a uint16 (2 bytes).

### Problem (transformer_accounting): Transformer LM resource accounting

#### Part (a)

Trainable parameters in the transformer LM: ~2 billion
Memory usage: ~8GB

In [12]:
import json

GPT2_XL = {
    "name": "GPT-2 XL",
    "vocab_size": 50527,
    "context_length": 1024,
    "num_layers": 48,
    "d_model": 1600,
    "num_heads": 25,
    "d_ff": 6400,
}

def analyze_params(config: dict) -> None:
    vocab_size = config["vocab_size"]
    context_length = config["context_length"]
    num_layers = config["num_layers"]
    d_model = config["d_model"]
    num_heads = config["num_heads"]
    d_ff = config["d_ff"]
    
    num_params = {
        "token_embedding": vocab_size * d_model,
        "transformer_block": {
            "rms_norm1": d_model,
            "multihead_self_attention": 4 * d_model * d_model,
            "rms_norm2": d_model,
            "SwiGLU": 3 * d_model * d_ff
        },
        "rms_norm_final": d_model,
        "linear_final": d_model * vocab_size,
    }

    total_params = (
        num_params["token_embedding"] + 
        num_layers * sum(num_params["transformer_block"].values()) + 
        num_params["rms_norm_final"] +
        num_params["linear_final"]
    )

    print(f"=== Param analysis for {config["name"]} === ")
    print(json.dumps(num_params, indent=4))
    print(f"Total: {total_params:,}")

analyze_params(GPT2_XL)

=== Param analysis for GPT-2 XL === 
{
    "token_embedding": 80843200,
    "transformer_block": {
        "rms_norm1": 1600,
        "multihead_self_attention": 10240000,
        "rms_norm2": 1600,
        "SwiGLU": 30720000
    },
    "rms_norm_final": 1600,
    "linear_final": 80843200
}
Total: 2,127,921,600


#### Part (b)
Matrix multiplications include the following. Total number of FLOPs required for one input sequence: ~4.5 trillion

In [13]:
def analyze_flops(config: dict) -> None:
    vocab_size = config["vocab_size"]
    context_length = config["context_length"]
    num_layers = config["num_layers"]
    d_model = config["d_model"]
    num_heads = config["num_heads"]
    d_ff = config["d_ff"]
    d_head = d_model // num_heads
    matmuls = {
        "transformer_block": {
            "multihead_self_attention": {
                "q_proj": 2 * context_length * d_model * d_model,
                "k_proj": 2 * context_length * d_model * d_model,
                "v_proj": 2 * context_length * d_model * d_model,
                "attention_scoring": num_heads * (2 * context_length * d_head * context_length),
                "applying_attention_weights": num_heads * (2 * context_length * context_length * d_head),
                "out_proj": 2 * context_length * d_model * d_model,
            },
            "SwiGLU": {
                "swiglu_w1_proj": 2 * context_length * d_model * d_ff,
                "swiglu_w3_proj": 2 * context_length * d_model * d_ff,
                "swiglu_w2_proj": 2 * context_length * d_model * d_ff,
            }
        },
        "final_linear": 2 * context_length * vocab_size * d_model
    }

    mhsa_flops = num_layers * sum(matmuls["transformer_block"]["multihead_self_attention"].values())
    swiglu_flops = num_layers * sum(matmuls["transformer_block"]["SwiGLU"].values())
    fl_flops = matmuls["final_linear"]
    total_flops = mhsa_flops + swiglu_flops + fl_flops

    print(f"=== Flops analysis for {config["name"]} ===")
    print(json.dumps(matmuls, indent=4))
    print(f"Total flops: {total_flops:,}")
    print(f"- MultiheadSelfAttention flops: {mhsa_flops:,} ({mhsa_flops/total_flops:.2%})")
    print(f"- SwiGLU flops: {swiglu_flops:,} ({swiglu_flops/total_flops:.2%})")
    print(f"- Final linear flops: {fl_flops:,} ({fl_flops/total_flops:.2%})")

analyze_flops(GPT2_XL)

=== Flops analysis for GPT-2 XL ===
{
    "transformer_block": {
        "multihead_self_attention": {
            "q_proj": 5242880000,
            "k_proj": 5242880000,
            "v_proj": 5242880000,
            "attention_scoring": 3355443200,
            "applying_attention_weights": 3355443200,
            "out_proj": 5242880000
        },
        "SwiGLU": {
            "swiglu_w1_proj": 20971520000,
            "swiglu_w3_proj": 20971520000,
            "swiglu_w2_proj": 20971520000
        }
    },
    "final_linear": 165566873600
}
Total flops: 4,514,221,260,800
- MultiheadSelfAttention flops: 1,328,755,507,200 (29.43%)
- SwiGLU flops: 3,019,898,880,000 (66.90%)
- Final linear flops: 165,566,873,600 (3.67%)


#### Part (c)

SwiGLU requires the most flops

#### Part (d)

As the size of the model increases, SwiGLU becomes more and more dominant, and final linear becomes almost negligbile.

In [14]:
GPT2_small = GPT2_XL.copy()
GPT2_small["name"] = "GPT-2 small"
GPT2_small["d_model"] = 768
GPT2_small["d_ff"] = int(GPT2_small["d_model"] * 8 / 3 / 64) * 64
GPT2_small["num_layers"] = 12
GPT2_small["num_heads"] = 12
analyze_flops(GPT2_small)

=== Flops analysis for GPT-2 small ===
{
    "transformer_block": {
        "multihead_self_attention": {
            "q_proj": 1207959552,
            "k_proj": 1207959552,
            "v_proj": 1207959552,
            "attention_scoring": 1610612736,
            "applying_attention_weights": 1610612736,
            "out_proj": 1207959552
        },
        "SwiGLU": {
            "swiglu_w1_proj": 3221225472,
            "swiglu_w3_proj": 3221225472,
            "swiglu_w2_proj": 3221225472
        }
    },
    "final_linear": 79472099328
}
Total flops: 292,072,980,480
- MultiheadSelfAttention flops: 96,636,764,160 (33.09%)
- SwiGLU flops: 115,964,116,992 (39.70%)
- Final linear flops: 79,472,099,328 (27.21%)


In [15]:
GPT2_medium = GPT2_XL.copy()
GPT2_medium["name"] = "GPT-2 medium"
GPT2_medium["d_model"] = 1024
GPT2_medium["d_ff"] = int(GPT2_medium["d_model"] * 8 / 3 / 64) * 64
GPT2_medium["num_layers"] = 24
GPT2_medium["num_heads"] = 16
analyze_flops(GPT2_medium)

=== Flops analysis for GPT-2 medium ===
{
    "transformer_block": {
        "multihead_self_attention": {
            "q_proj": 2147483648,
            "k_proj": 2147483648,
            "v_proj": 2147483648,
            "attention_scoring": 2147483648,
            "applying_attention_weights": 2147483648,
            "out_proj": 2147483648
        },
        "SwiGLU": {
            "swiglu_w1_proj": 5637144576,
            "swiglu_w3_proj": 5637144576,
            "swiglu_w2_proj": 5637144576
        }
    },
    "final_linear": 105962799104
}
Total flops: 821,074,853,888
- MultiheadSelfAttention flops: 309,237,645,312 (37.66%)
- SwiGLU flops: 405,874,409,472 (49.43%)
- Final linear flops: 105,962,799,104 (12.91%)


In [16]:
GPT2_large = GPT2_XL.copy()
GPT2_large["name"] = "GPT-2 large"
GPT2_large["d_model"] = 1280
GPT2_large["d_ff"] = int(GPT2_large["d_model"] * 8 / 3 / 64) * 64
GPT2_large["num_layers"] = 36
GPT2_large["num_heads"] = 20
analyze_flops(GPT2_large)

=== Flops analysis for GPT-2 large ===
{
    "transformer_block": {
        "multihead_self_attention": {
            "q_proj": 3355443200,
            "k_proj": 3355443200,
            "v_proj": 3355443200,
            "attention_scoring": 2684354560,
            "applying_attention_weights": 2684354560,
            "out_proj": 3355443200
        },
        "SwiGLU": {
            "swiglu_w1_proj": 8891924480,
            "swiglu_w3_proj": 8891924480,
            "swiglu_w2_proj": 8891924480
        }
    },
    "final_linear": 132453498880
}
Total flops: 1,769,238,691,840
- MultiheadSelfAttention flops: 676,457,349,120 (38.23%)
- SwiGLU flops: 960,327,843,840 (54.28%)
- Final linear flops: 132,453,498,880 (7.49%)


#### Part (e)

Total FLOPs increases to 150 trillion from 4 trillion. Now multihead self attention takes a much bigger proportion of total FLOPs.

In [17]:
GPT2_XL_long_context = GPT2_XL.copy()
GPT2_XL_long_context["context_length"] = 16384
analyze_flops(GPT2_XL_long_context)

=== Flops analysis for GPT-2 XL ===
{
    "transformer_block": {
        "multihead_self_attention": {
            "q_proj": 83886080000,
            "k_proj": 83886080000,
            "v_proj": 83886080000,
            "attention_scoring": 858993459200,
            "applying_attention_weights": 858993459200,
            "out_proj": 83886080000
        },
        "SwiGLU": {
            "swiglu_w1_proj": 335544320000,
            "swiglu_w3_proj": 335544320000,
            "swiglu_w2_proj": 335544320000
        }
    },
    "final_linear": 2649069977600
}
Total flops: 149,536,951,500,800
- MultiheadSelfAttention flops: 98,569,499,443,200 (65.92%)
- SwiGLU flops: 48,318,382,080,000 (32.31%)
- Final linear flops: 2,649,069,977,600 (1.77%)
